# PTC fit figure

Contact author: Alex Broughton
<br>Date: 


In [ ]:
! eups list -s | grep lsst_distrib

## Introduction

This notebook produces the three-panel PTC figure used in RTN-117. It shows measured covariances, the preliminary exponential model with saturation roll-off, the final full-covariance model, and fractional residuals for amplifier C03 on detector 94.

The output PDF is written to the current directory and copied into figures/ for the technote.


## 1.0 Set Up

In [ ]:
### Import packages and configure plotting defaults
from lsst.daf.butler import Butler
from lsst.ip.isr import PhotonTransferCurveDataset
from lsst.cp.pipe import PhotonTransferCurveSolveTask
from lsst.cp.pipe.utils import funcAstierWithRolloff
import matplotlib.pyplot as plt
import numpy as np

plt.rcParams.update({"font.size": 12})
import matplotlib as mpl
mpl.rcParams.update({
    "text.usetex": False,
    "font.family": "serif",
    "font.serif": ["CMU Serif", "Computer Modern Roman", "DejaVu Serif"],
})


butler = Butler("main")
camera = butler.get("camera", instrument="LSSTCam", collections="LSSTCam/defaults")

DETECTOR = 94
AMP = "C03"


## 2.0 Load calibration data

Load the final PTC calibration from the butler and the partial PTC datasets used to recompute the preliminary exponential fit.


In [ ]:
PTC_COLLECTION = "LSSTCam/calib/DM-53722/3s_v2/ptcGen.20260120a/20260122T004851Z"

partial_ptc_refs = butler.query_datasets(
    "cpPtcPartial", instrument="LSSTCam", detector=DETECTOR, collections=PTC_COLLECTION,
)
partial_ptcs = [butler.get(ref) for ref in partial_ptc_refs]

ptc = butler.get("ptc", instrument="LSSTCam", detector=DETECTOR, collections="LSSTCam/defaults")


## 2.1 Fit the preliminary model

Re-run the exponential approximation plus roll-off fit on the partial PTC datasets. This produces the green preliminary model curve without storing a hard-coded array.


In [ ]:
config = PhotonTransferCurveSolveTask.ConfigClass()
config.ptcFitType = "FULLCOVARIANCE"
config.maximumRangeCovariancesAstier = 10
config.maximumRangeCovariancesAstierFullCovFit = 10
config.sigmaCutPtcOutliers = 5.0
config.maxDeltaInitialPtcOutlierFit = 1_000
config.maxSignalInitialPtcOutlierFit = 20_000
config.extendRollofSearchMaskSizeAdu = 5_000
config.scaleMaxSignalInitialPtcOutlierFit = False
config.doModelPtcRolloff = True
config.maxPtcRolloffDeviation = 0.005

task = PhotonTransferCurveSolveTask(config=config)

amp_names = [amp.getName() for amp in camera[DETECTOR]]
assembled = PhotonTransferCurveDataset(ampNames=amp_names, ptcFitType=config.ptcFitType)
for partial in partial_ptcs:
    assembled.appendPartialPtc(partial)

prelim_ptc = task.fitPtc(assembled, computePtcTurnoff=True)
if config.doModelPtcRolloff:
    prelim_ptc = task.fitPtcRolloff(prelim_ptc)


def preliminary_c00(mu, amp):
    """Evaluate the preliminary C00 model (Eq. 16 + roll-off) in ADU^2."""
    pars = np.asarray(prelim_ptc.ptcFitPars[amp], dtype=float)
    mu_roll = prelim_ptc.ptcRolloff[amp]
    tau = prelim_ptc.ptcRolloffTau[amp]
    return funcAstierWithRolloff(np.r_[pars, mu_roll, tau], np.asarray(mu, dtype=float))


## 3.0 Triangle figure

Layout:

- upper left: C01
- lower left: C00
- lower right: C10
- upper right: empty


In [ ]:
XLIM = (0, 100_000)

domain = np.linspace(XLIM[0], XLIM[1], 10_000)
sub_domain = np.linspace(0, ptc.ptcTurnoff[AMP], 10_000)
pre_model = preliminary_c00(sub_domain, AMP)

# Create outer triangle layout
# -----------------------
fig = plt.figure(figsize=(14.0, 14.0), constrained_layout=False)

outer = fig.add_gridspec(
    2, 2,
    wspace=0.25,  # spacing between left/right panels
    hspace=0.25,  # spacing between top/bottom panels
)

# Each occupied cell gets its own 2-row subgridspec (top plot + residuals)
gs_fig3 = outer[0, 0].subgridspec(2, 1, height_ratios=[2, 1], hspace=0.05)  # upper-left
gs_fig1 = outer[1, 0].subgridspec(2, 1, height_ratios=[2, 1], hspace=0.05)  # lower-left
gs_fig2 = outer[1, 1].subgridspec(2, 1, height_ratios=[2, 1], hspace=0.05)  # lower-right

# Upper-right: empty
ax_empty = fig.add_subplot(outer[0, 1])
ax_empty.axis("off")

# Create axes (share x within each panel)
ax3_1 = fig.add_subplot(gs_fig3[0, 0])
ax3_2 = fig.add_subplot(gs_fig3[1, 0], sharex=ax3_1)

ax1_1 = fig.add_subplot(gs_fig1[0, 0])
ax1_2 = fig.add_subplot(gs_fig1[1, 0], sharex=ax1_1)

ax2_1 = fig.add_subplot(gs_fig2[0, 0])
ax2_2 = fig.add_subplot(gs_fig2[1, 0], sharex=ax2_1)

YLIM1 = (0, 55_000)

ax1_1.scatter(
    ptc.rawMeans[AMP],
    ptc.rawVars[AMP],
    s=30,
    c="k",
    edgecolor="w",
    alpha=0.25,
    label="All data",
)
ax1_1.scatter(
    ptc.finalMeans[AMP],
    ptc.finalVars[AMP],
    s=40,
    edgecolor="cornflowerblue",
    facecolor="none",
    label="Reserved",
)

ax1_1.plot(sub_domain, pre_model, "g--", label=r"$C_{00}^{\mathrm{prelim.\;model}}$")

ax1_1.plot(
    ptc.finalMeans[AMP],
    ptc.finalModelVars[AMP],
    color="darkorange",
    linewidth=2,
    label=r"$C_{i\;j}^{\mathrm{model}}$",
)

ax1_1.axvline(ptc.ptcTurnoff[AMP], linestyle="--", color="k")
ax1_1.axvline(ptc.ptcRolloff[AMP], linestyle="--", color="k")

ax1_1.plot(domain, domain * (1 / ptc.gain[AMP]), "k-", alpha=0.25)
ax1_1.text(0.6e5, 4.7e4, r"$1\;/\;g$", rotation=40., alpha=0.5)

ax1_1.text(ptc.ptcRolloff[AMP] - 3000, 5000, "PTC Roll-off", rotation=90)
ax1_1.text(ptc.ptcTurnoff[AMP] - 3000, 5000, "PTC Turn-off", rotation=90)

ax1_1.legend()
ax1_1.set_ylabel(r"$C_{00}$ (ADU$^2$)")
ax1_1.set_xlim(XLIM)
ax1_1.set_ylim(YLIM1)
ax1_1.ticklabel_format(style="sci", axis="x", scilimits=(0, 0), useMathText=True)
ax1_1.ticklabel_format(style="sci", axis="y", scilimits=(0, 0), useMathText=True)
ax1_1.tick_params(labelbottom=False)

# Residuals (Figure 1 bottom)
ax1_2.axhline(0, color="darkorange", linewidth=2)

ax1_2.scatter(
    ptc.rawMeans[AMP],
    (ptc.evalPtcModel(ptc.rawMeans[AMP])[AMP][:, 0, 0] - ptc.rawVars[AMP]) / ptc.rawVars[AMP],
    s=30,
    c="k",
    edgecolor="w",
    alpha=0.25,
)
ax1_2.scatter(
    ptc.finalMeans[AMP],
    (ptc.evalPtcModel(ptc.finalMeans[AMP])[AMP][:, 0, 0] - ptc.finalVars[AMP]) / ptc.finalVars[AMP],
    s=40,
    edgecolor="cornflowerblue",
    facecolor="none",
)

ax1_2.axvline(ptc.ptcTurnoff[AMP], linestyle="--", color="k")
ax1_2.axvline(ptc.ptcRolloff[AMP], linestyle="--", color="k")

ax1_2.set_ylim(-4e-2, 4e-2)
ax1_2.set_xlim(XLIM)
ax1_2.ticklabel_format(style="sci", axis="x", scilimits=(0, 0), useMathText=True)
ax1_2.set_xlabel(r"Flat field level, $\mu$ (ADU)")
ax1_2.set_ylabel(r"$C_{00} / C_{00}^{\mathrm{model}} - 1$")

YLIM2 = (0, 4_000)
expIdMask = ptc.expIdMask[AMP]

ax2_1.scatter(
    ptc.rawMeans[AMP],
    ptc.covariances[AMP][:, 1, 0],
    s=30,
    c="k",
    edgecolor="w",
    alpha=0.25,
)
ax2_1.scatter(
    ptc.rawMeans[AMP][expIdMask],
    ptc.covariances[AMP][expIdMask][:, 1, 0],
    s=40,
    edgecolor="cornflowerblue",
    facecolor="none",
)

ax2_1.plot(
    ptc.rawMeans[AMP][expIdMask],
    ptc.evalPtcModel(ptc.rawMeans[AMP][expIdMask])[AMP][:, 1, 0],
    color="darkorange",
    linewidth=2,
)

ax2_1.axvline(ptc.ptcTurnoff[AMP], linestyle="--", color="k")
ax2_1.axvline(ptc.ptcRolloff[AMP], linestyle="--", color="k")

ax2_1.text(ptc.ptcRolloff[AMP] - 3000, 1.0e3, "PTC Roll-off", rotation=90)
ax2_1.text(ptc.ptcTurnoff[AMP] - 3000, 1.0e3, "PTC Turn-off", rotation=90)

ax2_1.set_ylabel(r"$C_{10}$ (ADU$^2$)")
ax2_1.set_xlim(XLIM)
ax2_1.set_ylim(YLIM2)
ax2_1.ticklabel_format(style="sci", axis="x", scilimits=(0, 0), useMathText=True)
ax2_1.ticklabel_format(style="sci", axis="y", scilimits=(0, 0), useMathText=True)
ax2_1.tick_params(labelbottom=False)

# Residuals (Figure 2 bottom)
ax2_2.axhline(0, color="darkorange", linewidth=2)

ax2_2.scatter(
    ptc.rawMeans[AMP],
    (ptc.covariances[AMP][:, 1, 0] - ptc.evalPtcModel(ptc.rawMeans[AMP])[AMP][:, 1, 0])
    / ptc.evalPtcModel(ptc.rawMeans[AMP])[AMP][:, 1, 0],
    s=30,
    c="k",
    edgecolor="w",
    alpha=0.25,
)
ax2_2.scatter(
    ptc.rawMeans[AMP][expIdMask],
    (ptc.covariances[AMP][expIdMask][:, 1, 0] - ptc.evalPtcModel(ptc.rawMeans[AMP][expIdMask])[AMP][:, 1, 0])
    / ptc.evalPtcModel(ptc.rawMeans[AMP][expIdMask])[AMP][:, 1, 0],
    s=40,
    edgecolor="cornflowerblue",
    facecolor="none",
)

ax2_2.set_ylim(-2.5, 2.5)
ax2_2.set_xlim(XLIM)
ax2_2.ticklabel_format(style="sci", axis="x", scilimits=(0, 0), useMathText=True)
ax2_2.set_xlabel(r"Flat field level, $\mu$ (ADU)")
ax2_2.set_ylabel(r"$C_{10} / C_{10}^{\mathrm{model}} - 1$")
ax2_2.axvline(ptc.ptcTurnoff[AMP], linestyle="--", color="k")
ax2_2.axvline(ptc.ptcRolloff[AMP], linestyle="--", color="k")

YLIM3 = (0, 4_000)
expIdMask = ptc.expIdMask[AMP]

ax3_1.scatter(
    ptc.rawMeans[AMP],
    ptc.covariances[AMP][:, 0, 1],
    s=30,
    c="k",
    edgecolor="w",
    alpha=0.25,
)
ax3_1.scatter(
    ptc.rawMeans[AMP][expIdMask],
    ptc.covariances[AMP][expIdMask][:, 0, 1],
    s=40,
    edgecolor="cornflowerblue",
    facecolor="none",
)

ax3_1.plot(
    ptc.rawMeans[AMP][expIdMask],
    ptc.evalPtcModel(ptc.rawMeans[AMP][expIdMask])[AMP][:, 0, 1],
    color="darkorange",
    linewidth=2,
)

ax3_1.axvline(ptc.ptcTurnoff[AMP], linestyle="--", color="k")
ax3_1.axvline(ptc.ptcRolloff[AMP], linestyle="--", color="k")

ax3_1.text(ptc.ptcRolloff[AMP] - 3000, 0.35e3, "PTC Roll-off", rotation=90)
ax3_1.text(ptc.ptcTurnoff[AMP] - 3000, 0.35e3, "PTC Turn-off", rotation=90)

ax3_1.set_ylabel(r"$C_{01}$ (ADU$^2$)")
ax3_1.set_xlim(XLIM)
ax3_1.set_ylim(YLIM3)
ax3_1.ticklabel_format(style="sci", axis="x", scilimits=(0, 0), useMathText=True)
ax3_1.ticklabel_format(style="sci", axis="y", scilimits=(0, 0), useMathText=True)
ax3_1.tick_params(labelbottom=False)

# Residuals (Figure 3 bottom)
ax3_2.axhline(0, color="darkorange", linewidth=2)

ax3_2.scatter(
    ptc.rawMeans[AMP],
    (ptc.covariances[AMP][:, 0, 1] - ptc.evalPtcModel(ptc.rawMeans[AMP])[AMP][:, 0, 1])
    / ptc.evalPtcModel(ptc.rawMeans[AMP])[AMP][:, 0, 1],
    s=30,
    c="k",
    edgecolor="w",
    alpha=0.25,
)
ax3_2.scatter(
    ptc.rawMeans[AMP][expIdMask],
    (ptc.covariances[AMP][expIdMask][:, 0, 1] - ptc.evalPtcModel(ptc.rawMeans[AMP][expIdMask])[AMP][:, 0, 1])
    / ptc.evalPtcModel(ptc.rawMeans[AMP][expIdMask])[AMP][:, 0, 1],
    s=40,
    edgecolor="cornflowerblue",
    facecolor="none",
)

ax3_2.set_ylim(-2.5, 2.5)
ax3_2.set_xlim(XLIM)
ax3_2.ticklabel_format(style="sci", axis="x", scilimits=(0, 0), useMathText=True)
ax3_2.set_xlabel(r"Flat field level, $\mu$ (ADU)")
ax3_2.set_ylabel(r"$C_{01} / C_{01}^{\mathrm{model}} - 1$")
ax3_2.axvline(ptc.ptcTurnoff[AMP], linestyle="--", color="k")
ax3_2.axvline(ptc.ptcRolloff[AMP], linestyle="--", color="k")

ax1_1.set_title(r"$(0,0)$")
ax2_1.set_title(r"$(1,0)$")
ax3_1.set_title(r"$(0,1)$")

plt.savefig(f"ptc-triangle-det{DETECTOR}-amp{AMP}.pdf", format="pdf", bbox_inches="tight")
plt.show()